In [4]:
# Homework 2

# Problem 2
import pandas as pd
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score

# Load datasets
train_df = pd.read_csv("train.csv")
test_df = pd.read_csv("test.csv")

# Columns to exclude
drop_cols = ["id", "date", "zipcode", "Unnamed: 0"]

# Drop excluded columns safely
train_drop = [c for c in drop_cols if c in train_df.columns]
test_drop  = [c for c in drop_cols if c in test_df.columns]

# Separate features and target (training)
X_train = train_df.drop(columns=train_drop + ["price"])
y_train = train_df["price"] / 1000.0

# Prepare test features
has_test_labels = "price" in test_df.columns
X_test = test_df.drop(columns=test_drop + (["price"] if has_test_labels else []))
if has_test_labels:
    y_test = test_df["price"] / 1000.0

# Align feature columns
X_test = X_test.reindex(columns=X_train.columns)

# Handle missing values (mean imputation)
imputer = SimpleImputer(strategy="mean")
X_train_imp = imputer.fit_transform(X_train)
X_test_imp = imputer.transform(X_test)

# Standardize features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_imp)
X_test_scaled = scaler.transform(X_test_imp)

# Train linear regression model
lr = LinearRegression()
lr.fit(X_train_scaled, y_train)

# Training predictions and metrics
y_train_pred = lr.predict(X_train_scaled)
train_mse = mean_squared_error(y_train, y_train_pred)
train_r2 = r2_score(y_train, y_train_pred)

print("Training MSE:", train_mse)
print("Training R^2:", train_r2)

# Report coefficients
coef_df = pd.DataFrame({
    "feature": X_train.columns,
    "coefficient": lr.coef_
}).sort_values(by="coefficient", ascending=False)

print("\nIntercept:", lr.intercept_)
print("\nModel Coefficients:")
print(coef_df.to_string(index=False))

y_test_pred = lr.predict(X_test_scaled)

if has_test_labels:
    test_mse = mean_squared_error(y_test, y_test_pred)
    test_r2 = r2_score(y_test, y_test_pred)
    print("\nTesting MSE:", test_mse)
    print("Testing R^2:", test_r2)
else:
    print("\nTest set does not include price labels.")
    print("Predictions stored in y_test_pred.")

top_features = (
    coef_df.assign(abs_coef=coef_df["coefficient"].abs())
           .sort_values("abs_coef", ascending=False)
           .drop(columns="abs_coef")
           .head(10)
)

print("\nTop 10 contributing features (by |coefficient|):")
print(top_features.to_string(index=False))


Training MSE: 31486.16777579488
Training R^2: 0.7265334318706018

Intercept: 520.414834000001

Model Coefficients:
      feature  coefficient
        grade    92.231475
          lat    78.375737
   waterfront    63.742900
  sqft_living    56.748837
   sqft_above    48.290089
         view    48.200109
sqft_living15    45.577658
sqft_basement    27.137032
    bathrooms    18.527633
 yr_renovated    17.271380
    condition    12.964269
     sqft_lot    10.881868
       floors     8.043721
         long    -1.035203
     bedrooms   -12.521962
   sqft_lot15   -12.930091
     yr_built   -67.643117

Test set does not include price labels.
Predictions stored in y_test_pred.

Top 10 contributing features (by |coefficient|):
      feature  coefficient
        grade    92.231475
          lat    78.375737
     yr_built   -67.643117
   waterfront    63.742900
  sqft_living    56.748837
   sqft_above    48.290089
         view    48.200109
sqft_living15    45.577658
sqft_basement    27.137032
   

In [6]:
# Problem 3
# Columns to drop
drop_cols = ["id", "date", "zipcode", "Unnamed: 0"]
train_drop = [c for c in drop_cols if c in train_df.columns]
test_drop = [c for c in drop_cols if c in test_df.columns]

# Split features and target
X_train = train_df.drop(columns=train_drop + ["price"])
y_train = train_df["price"].values.reshape(-1, 1) / 1000.0

X_test = test_df.drop(columns=test_drop + ["price"], errors="ignore")
has_test_labels = "price" in test_df.columns
if has_test_labels:
    y_test = test_df["price"].values.reshape(-1, 1) / 1000.0

# Align columns
X_test = X_test.reindex(columns=X_train.columns)

# Handle missing values
imputer = SimpleImputer(strategy="mean")
X_train_imp = imputer.fit_transform(X_train)
X_test_imp = imputer.transform(X_test)

# Standardize features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_imp)
X_test_scaled = scaler.transform(X_test_imp)

# Add bias (intercept) term
X_train_cf = np.hstack([np.ones((X_train_scaled.shape[0], 1)), X_train_scaled])
X_test_cf = np.hstack([np.ones((X_test_scaled.shape[0], 1)), X_test_scaled])

theta = np.linalg.inv(X_train_cf.T @ X_train_cf) @ X_train_cf.T @ y_train

def predict_closed_form(X, theta):
    return X @ theta

y_train_pred_cf = predict_closed_form(X_train_cf, theta)
train_mse_cf = mean_squared_error(y_train, y_train_pred_cf)
train_r2_cf = r2_score(y_train, y_train_pred_cf)

print("Closed-Form Training MSE:", train_mse_cf)
print("Closed-Form Training R^2:", train_r2_cf)

if has_test_labels:
    y_test_pred_cf = predict_closed_form(X_test_cf, theta)
    test_mse_cf = mean_squared_error(y_test, y_test_pred_cf)
    test_r2_cf = r2_score(y_test, y_test_pred_cf)
    print("Closed-Form Testing MSE:", test_mse_cf)
    print("Closed-Form Testing R^2:", test_r2_cf)

lr = LinearRegression()
lr.fit(X_train_scaled, y_train)

y_train_pred_pkg = lr.predict(X_train_scaled)
pkg_train_mse = mean_squared_error(y_train, y_train_pred_pkg)
pkg_train_r2 = r2_score(y_train, y_train_pred_pkg)

print("\nSklearn Training MSE:", pkg_train_mse)
print("Sklearn Training R^2:", pkg_train_r2)

Closed-Form Training MSE: 1140650.5115485461
Closed-Form Training R^2: -8.906883017628479

Sklearn Training MSE: 31486.16777579488
Sklearn Training R^2: 0.7265334318706018


In [9]:
# Problem 4
from sklearn.model_selection import train_test_split

# Load Data
df = pd.read_csv("kc_house_data.csv")

# Feature and target
X = df[["sqft_living"]].values
y = df["price"].values / 1000  # divide price by 1000

# Train test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Standardize X
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

def polynomial_features(X, degree):
    """
    Create polynomial features [1, X, X^2, ..., X^degree]
    """
    X_poly = np.ones((X.shape[0], 1))
    for d in range(1, degree + 1):
        X_poly = np.hstack((X_poly, X ** d))
    return X_poly

# Closed form linear regression

def closed_form_train(X, y):
    return np.linalg.inv(X.T @ X) @ X.T @ y

def predict(X, theta):
    return X @ theta

# Train models for different p values

results = []

for p in [1, 2, 3, 5]:
    X_train_poly = polynomial_features(X_train, p)
    X_test_poly = polynomial_features(X_test, p)

    theta = closed_form_train(X_train_poly, y_train)

    y_train_pred = predict(X_train_poly, theta)
    y_test_pred = predict(X_test_poly, theta)

    results.append({
        "degree (p)": p,
        "train MSE": mean_squared_error(y_train, y_train_pred),
        "train R^2": r2_score(y_train, y_train_pred),
        "test MSE": mean_squared_error(y_test, y_test_pred),
        "test R^2": r2_score(y_test, y_test_pred)
    })

results_df = pd.DataFrame(results)
print(results_df)


   degree (p)     train MSE  train R^2      test MSE  test R^2
0           1  66319.347785   0.492384  76484.977062  0.494069
1           2  58871.855127   0.549388  82113.931184  0.456835
2           3  58862.529017   0.549459  83663.526745  0.446585
3           5  58812.805203   0.549840  85501.107850  0.434429


In [12]:
# Problem 5

# Question 1
def add_bias(X):
    return np.hstack([np.ones((X.shape[0], 1)), X])

def predict(Xb, theta):
    return Xb @ theta

def gradient_descent(Xb, y, alpha, num_iters, theta_init=None):
    N, d = Xb.shape
    y = y.reshape(-1, 1)

    if theta_init is None:
        theta = np.zeros((d, 1))
    else:
        theta = theta_init.reshape(-1, 1).copy()

    for _ in range(num_iters):
        errors = (Xb @ theta) - y
        grad = (2.0 / N) * (Xb.T @ errors)
        theta = theta - alpha * grad

    return theta

# Question 2
X_train_sc, y_train, X_test_sc, y_test = load_and_preprocess()

Xb_train = add_bias(X_train_sc)
Xb_test = add_bias(X_test_sc)

alphas = [0.01, 0.1, 0.5]
iters_list = [10, 50, 100]

rows = []

for alpha in alphas:
    for iters in iters_list:
        theta = gradient_descent(Xb_train, y_train, alpha=alpha, num_iters=iters)

        # Predictions
        yhat_train = predict(Xb_train, theta).ravel()
        train_mse = mean_squared_error(y_train, yhat_train)
        train_r2 = r2_score(y_train, yhat_train)

        # Test metrics only if test labels exist
        if y_test is not None:
            yhat_test = predict(Xb_test, theta).ravel()
            test_mse = mean_squared_error(y_test, yhat_test)
            test_r2 = r2_score(y_test, yhat_test)
        else:
            test_mse, test_r2 = np.nan, np.nan

        # Store theta 
        theta_preview = np.array2string(theta.ravel()[:6], precision=4, separator=", ")

        rows.append({
            "alpha": alpha,
            "iters": iters,
            "theta (first 6)": theta_preview,
            "train MSE": train_mse,
            "train R^2": train_r2,
            "test MSE": test_mse,
            "test R^2": test_r2
        })

results_df = pd.DataFrame(rows)
print(results_df.to_string(index=False))

# Question 3
# Algorithm converges
def closed_form_theta(Xb, y):
    y = y.reshape(-1, 1)
    # Use pseudo inverse for numerical stability
    return np.linalg.pinv(Xb) @ y

theta_star = closed_form_theta(Xb_train, y_train)
print("\nClosed-form theta (first 6):", np.array2string(theta_star.ravel()[:6], precision=4, separator=", "))

 alpha  iters                                                                  theta (first 6)     train MSE      train R^2  test MSE  test R^2
  0.01     10                           [95.198 , 11.9286, 20.2833, 32.1165,  5.3808,  9.5255]  2.357278e+05  -1.047365e+00       NaN       NaN
  0.01     50                     [330.8955,   6.0053,  23.8214,  54.6659,   3.6375,  11.122 ]  6.972050e+04   3.944571e-01       NaN       NaN
  0.01    100                     [451.3976,  -3.6569,  19.2418,  56.955 ,   2.6937,  10.4482]  3.682035e+04   6.802045e-01       NaN       NaN
  0.10     10                     [464.5357,  -4.6456,  18.7334,  57.1255,   2.53  ,  10.5525]  3.510510e+04   6.951019e-01       NaN       NaN
  0.10     50                     [520.4074, -12.4572,  17.6212,  57.1878,   7.8533,   8.006 ]  3.149726e+04   7.264371e-01       NaN       NaN
  0.10    100                     [520.4148, -12.5191,  18.4242,  56.7696,  10.2707,   8.061 ]  3.148643e+04   7.265311e-01       NaN   

In [20]:
# Problem 6
# Question 2

def add_bias(X):
    return np.hstack([np.ones((X.shape[0], 1)), X])

def predict(Xb, theta):
    return (Xb @ theta).ravel()

def ridge_gradient_descent(Xb, y, alpha, num_iters, lam, theta_init=None):
    N, d = Xb.shape
    y = y.reshape(-1, 1)

    # Initialize theta
    if theta_init is None:
        theta = np.zeros((d, 1))
    else:
        theta = np.array(theta_init, dtype=float).reshape(-1, 1)

    # Gradient descent updates
    for _ in range(num_iters):
        # Prediction errors
        errors = (Xb @ theta) - y

        # Gradient of MSE term: (2/N) * X^T (X theta - y)
        grad_mse = (2.0 / N) * (Xb.T @ errors)

        # Gradient of ridge term: 2*lam*theta, but do NOT penalize intercept theta[0]
        grad_reg = 2.0 * lam * theta
        grad_reg[0] = 0.0

        # Update
        theta = theta - alpha * (grad_mse + grad_reg)

    return theta

In [21]:
# Problem 6 
# Question 3

np.random.seed(42)

# Simulate data
N = 1000
X = np.random.uniform(-2, 2, size=(N, 1))
e = np.random.normal(loc=0.0, scale=np.sqrt(2), size=(N, 1))  # Var=2
y = 1 + 2*X + e

# Build design matrix with bias term
Xb = np.hstack([np.ones((N, 1)), X])

def ridge_closed_form(Xb, y, lam):
    d = Xb.shape[1]
    I = np.eye(d)
    I[0, 0] = 0.0
    theta = np.linalg.inv(Xb.T @ Xb + lam * I) @ (Xb.T @ y)
    return theta

lams = [0, 1, 10, 100, 1000, 10000]  

rows = []
for lam in lams:
    theta = ridge_closed_form(Xb, y, lam)
    yhat = Xb @ theta

    mse = mean_squared_error(y, yhat)
    r2 = r2_score(y, yhat)

    intercept = float(theta[0])
    slope = float(theta[1])

    rows.append((lam, intercept, slope, mse, r2))

# Print results
print("lambda   intercept     slope        MSE        R^2")
for lam, b0, b1, mse, r2 in rows:
    print(f"{lam:<7}  {b0:>10.4f}   {b1:>9.4f}   {mse:>9.4f}   {r2:>7.4f}")

lambda   intercept     slope        MSE        R^2
0            1.1377      1.9453      1.9499    0.7258
1            1.1377      1.9439      1.9499    0.7258
10           1.1372      1.9311      1.9502    0.7258
100          1.1325      1.8124      1.9740    0.7224
1000         1.1057      1.1225      2.8735    0.5960
10000        1.0710      0.2335      5.9471    0.1638


/var/folders/yq/hhnlxf4112jbmhsjv76vc3g40000gn/T/ipykernel_42904/795925231.py:36: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  intercept = float(theta[0])
/var/folders/yq/hhnlxf4112jbmhsjv76vc3g40000gn/T/ipykernel_42904/795925231.py:37: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  slope = float(theta[1])
/var/folders/yq/hhnlxf4112jbmhsjv76vc3g40000gn/T/ipykernel_42904/795925231.py:36: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  intercept = float(theta[0])
/var/